# Managing Kleio files

> First time use: follow instructions in the [README.md](README.md) file in this directory.

## Initial setup

Create a TimelinkNotebook object. This will be the interface for most of your interactions with Timelink.

The first time you run this notebook, you will be prompted to install ìpykernel. This is required to run Timelink in a notebook.

Note:
* First time run takes a little time as the required Docker images are downloaded.
* Timelink will default to using sqlite as the database, see [Receipts notebook for more control](1-receipts.ipynb)


In [ ]:
from timelink.notebooks import TimelinkNotebook
from timelink.kleio import KleioServer

# Alternative: attach to existing Kleio server for debugging
# kserver = KleioServer.attach(url='http://127.0.0.1:8088',token='mytoken',kleio_home='../sources')
# tlnb = TimelinkNotebook(kleio_server=kserver)
tlnb = TimelinkNotebook(kleio_image="kleio-server", 
                        kleio_version="12.9.592",
                        kleio_update=True,)

tlnb.print_info()


TimelinkNotebook created a database to store data and a Kleio server to 
translate transcriptions in Kleio notation into data that can be imported
into a database.

To get information about the database and the Kleio server do ```tnlb.print_info()```

### Database status

Count the number of rows in each table in the database.


In [ ]:
tlnb.table_row_count_df()

### Show the kleio files available 

In [ ]:
kleio_files = tlnb.get_kleio_files(match_path=True)
# kleio_files.info()
kleio_files[['name','status','errors','warnings','import_status','import_errors']].sort_values("name")  


## Optionally clean translations 

Do this if there was an update to the Kleio server and you want to make sure that all translations are redone to incorporate the changes.

In [ ]:
# NBVAL_IGNORE_OUTPUT

# tlnb.kleio_server.translation_clean("", recurse="yes")


### Translate and import as needed

In [ ]:
tlnb.db.update_from_sources(path='', with_import_errors=True, match_path=True)
kleio_files = tlnb.get_kleio_files()
kleio_files[['name','status','errors','warnings','import_status','import_errors']]


## Check the status of the files

Check the import status of the translated files

* I: Imported
* E: Imported with error
* W: Imported with warnings no errors
* N: Not imported
* U: Translation updated need to reimport

### Check a translation report

If needed to check for errors.


In [ ]:
# get the translation status
imported_files_df = tlnb.get_import_status(match_path=True)
errors = imported_files_df[imported_files_df["errors"] > 0 ]
errors[["path","name", "errors", "warnings", "import_status", "import_errors", "import_warnings", "path"]].sort_values("name")

In [ ]:
imported_files_df[['name','import_status']]

In [ ]:
file_number=list(errors.index.unique())
if len(file_number) > 0:
    rpt = tlnb.get_translation_report(imported_files_df, file_number)
    print(rpt)
else:
    print("No errors found")



### Get the import report for a file

In case there are errors in the import phase check the import report.

In [ ]:
ierrors = imported_files_df[imported_files_df.import_errors>0][["import_status", "import_errors","import_warnings","name","imported","errors","warnings","path"]].sort_values("name")
ierrors

In [ ]:
for ierror in ierrors.index:
    print(f"File: {ierrors.loc[ierror,'name']} - {ierrors.loc[ierror,'path']}")
    rpt = tlnb.get_import_rpt(imported_files_df, rows=ierror, match_path=True)
    print(rpt)

In [ ]:
tlnb.get_import_rpt('dehergne-v.cli')  # just the file name of the kleio file

In [ ]:
from datetime import datetime

# Print current timestamp
print(datetime.now())

# Or with a specific format
print(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

# Or using other formats
print(f"Current time: {datetime.now()}")
